# 01 — Extração de papers das três fontes

Notebook que executa os **filtros 1 e 2** do pipeline (ver `00_design.ipynb`):

- **Filtro 1 (descoberta):** queries de keywords em três APIs públicas (arXiv, OpenAlex e Semantic Scholar).
- **Filtro 2 (ranking):** codificamos `title + abstract` com um modelo de embedding leve e ranqueamos por similaridade a queries-âncora que descrevem o paper-tipo que queremos extrair.

**Saída:** `data/processed/papers_ranked.parquet`: top-K papers ordenados por relevância, prontos para o classificador LLM no próximo notebook.

## Estratégia em três passos

1. **Coletar** com queries amplas em cada fonte (over-recall na descoberta).
2. **Unificar e deduplicar** pelos identificadores (arXiv ID > DOI > título normalizado).
3. **Ranquear** por embedding semântico contra queries-âncora; reter top-K.

In [ ]:
!pip install arxiv sentence-transformers scikit-learn pyarrow tqdm requests pandas

In [1]:
import os
import re
import time
import json
import hashlib
from pathlib import Path

import requests
import pandas as pd
from tqdm.auto import tqdm

c:\Users\fredb\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NB_DIR = Path.cwd()
PROJECT_DIR = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

USER_EMAIL = "" # Adicione aqui. Removido para o commit

TOP_K = 3000

## Definição das queries

- `KEYWORD_QUERIES`: termos curtos para busca textual nas APIs. Cobrem o espaço de "imbalanced learning" sem ser excessivamente específicos (a ideia aqui é ter recall alto - é mais barato descartar falsos-positivos depois do que perder papers relevantes aqui).
- `EMBEDDING_QUERIES`: descrições mais longas que representam o **paper-alvo ideal**. Usadas no filtro 2 para ranquear por similaridade semântica. Capturam relevância que keywords não capturam (ex: paper que faz a coisa certa mas com terminologia atípica).

In [4]:
KEYWORD_QUERIES = [
    "class imbalance",
    "imbalanced classification",
    "imbalanced learning",
    "imbalanced data",
    "minority class oversampling",
    "undersampling minority class",
    "cost-sensitive learning",
    "class reweighting",
    "class-balanced loss",
    "SMOTE",
    "ADASYN",
    "focal loss",
    "long-tail recognition",
    "long-tailed classification",
]

EMBEDDING_QUERIES = [
    # Q1: Benchmark comparativo
    "Empirical comparison of class balancing strategies (SMOTE, ADASYN, random oversampling, undersampling, class weights, focal loss) for imbalanced classification, reporting macro-F1, balanced accuracy, AUPRC, or per-class TPR / TPR gap on test data.",
    # Q2: Aplicação com baseline explícito
    "Supervised classification on imbalanced data: experimental evaluation of a resampling, augmentation, or reweighting strategy against a no-intervention baseline, with per-class metrics (recall, F1, TPR) reported.",
    # Q3: Method paper (novo método sendo proposto)
    "Novel resampling, reweighting, or cost-sensitive method for imbalanced or long-tailed classification, benchmarked against existing strategies with macro-F1, balanced accuracy, or sensitivity / TPR gap.",
    # Q4: Aplicação em domínio
    "Class imbalance in real-world classification tasks such as fraud detection, medical diagnosis, anomaly detection, or long-tail visual recognition, with comparison of oversampling, undersampling, augmentation, or loss-based approaches.",
]

## Fonte 1: arXiv

Usamos a biblioteca `arxiv`. Restringimos às categorias relevantes de ML/CV/NLP (`cs.LG`, `cs.AI`, `stat.ML`, `cs.CV`, `cs.CL`).

Cada query retorna até `max_per_query` resultados ordenados por relevância. Saída raw em `data/raw/arxiv_papers.parquet`.

In [6]:
import arxiv

ARXIV_CATEGORIES = "(cat:cs.LG OR cat:cs.AI OR cat:stat.ML OR cat:cs.CV OR cat:cs.CL)"

def search_arxiv(queries, max_per_query=800, page_size=200, delay_seconds=3.0):
    client = arxiv.Client(page_size=page_size, delay_seconds=delay_seconds, num_retries=3)
    rows = []
    for q in queries:
        full_query = f"({q}) AND {ARXIV_CATEGORIES}"
        search = arxiv.Search(query=full_query, max_results=max_per_query, sort_by=arxiv.SortCriterion.Relevance)
        try:
            count = 0
            for r in client.results(search):
                arxiv_id = r.entry_id.split("/abs/")[-1]
                rows.append({
                    "arxiv_id": arxiv_id,
                    "doi": r.doi,
                    "title": (r.title or "").strip().replace("\n", " "),
                    "abstract": (r.summary or "").strip().replace("\n", " "),
                    "authors": [a.name for a in r.authors],
                    "year": r.published.year if r.published else None,
                    "venue": None,
                    "categories": r.categories,
                    "oa_pdf_url": r.pdf_url,
                    "is_oa": True,
                    "discovered_via": "arxiv",
                    "discovered_query": q,
                })
                count += 1
            print(f"'{q}': {count} resultados")
        except Exception as e:
            print(f"'{q}': ERRO — {type(e).__name__}: {e}")
    return pd.DataFrame(rows)

In [7]:
arxiv_path = RAW_DIR / "arxiv_papers.parquet"

if arxiv_path.exists():
    print(f"Cache: {arxiv_path}")
    df_arxiv = pd.read_parquet(arxiv_path)
else:
    print("Buscando no arXiv...")
    df_arxiv = search_arxiv(KEYWORD_QUERIES, max_per_query=800)
    df_arxiv.to_parquet(arxiv_path, index=False)
    print(f"Salvo em {arxiv_path}")

print(f"\nTotal de linhas (com duplicatas entre queries): {len(df_arxiv)}")
print(f"Únicos por arxiv_id: {df_arxiv['arxiv_id'].nunique()}")
df_arxiv.head(3)

Buscando no arXiv...
'class imbalance': 800 resultados
'imbalanced classification': 800 resultados
'imbalanced learning': 800 resultados
'imbalanced data': 800 resultados
'minority class oversampling': 800 resultados
'undersampling minority class': 800 resultados
'cost-sensitive learning': 800 resultados
'class reweighting': 800 resultados
'class-balanced loss': 800 resultados
'SMOTE': 311 resultados
'ADASYN': 26 resultados
'focal loss': 800 resultados
'long-tail recognition': 800 resultados
'long-tailed classification': 800 resultados
Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\raw\arxiv_papers.parquet

Total de linhas (com duplicatas entre queries): 9937
Únicos por arxiv_id: 5447


,arxiv_id,doi,title,abstract,authors,year,venue,categories,oa_pdf_url,is_oa,discovered_via,discovered_query
0,1809.10388v2,10.1007/978-3-030-01418-6_49,Queue-based Resampling for Online Class Imbala...,Online class imbalance learning constitutes a ...,"[Kleanthis Malialis, Christos G. Panayiotou, M...",2018,None,"[cs.LG, stat.ML]",https://arxiv.org/pdf/1809.10388v2,True,arxiv,class imbalance
1,1307.5730v1,None,A New Strategy of Cost-Free Learning in the Cl...,"In this work, we define cost-free learning (CF...","[Xiaowan Zhang, Bao-Gang Hu]",2013,None,[cs.LG],https://arxiv.org/pdf/1307.5730v1,True,arxiv,class imbalance
2,2008.05524v1,None,Mitigating Dataset Imbalance via Joint Generat...,Supervised deep learning methods are enjoying ...,"[Aadarsh Sahoo, Ankit Singh, Rameswar Panda, R...",2020,None,[cs.CV],https://arxiv.org/pdf/2008.05524v1,True,arxiv,class imbalance


## Fonte 2: OpenAlex

API gratuita que Cobre venues que não publicam pré-print no arXiv (IEEE, Elsevier...)

Filtros aplicados:
- `type:article` (exclui datasets, livros, etc.)
- `has_abstract:true` (precisa de abstract para o filtro de embedding)

Abstracts no OpenAlex vêm como `abstract_inverted_index` (dicionário de `word` para `posições`). Reconstruímos a string.

In [8]:
def _reconstruct_abstract_oa(inv_idx):
    if not inv_idx:
        return ""
    pos_to_word = {}
    for word, positions in inv_idx.items():
        for p in positions:
            pos_to_word[p] = word
    if not pos_to_word:
        return ""
    return " ".join(pos_to_word[i] for i in sorted(pos_to_word))

def _normalize_doi(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    x = str(x).lower().strip()
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x)
    return x or None

def search_openalex(queries, per_page=200, max_per_query=1500, email=None):
    base = "https://api.openalex.org/works"
    headers = {"User-Agent": f"causal-ml-project ({email})"}
    rows = []
    for q in queries:
        cursor = "*"
        collected = 0
        while collected < max_per_query:
            params = {
                "search": q,
                "per_page": per_page,
                "cursor": cursor,
                "filter": "type:article,has_abstract:true",
                "select": "id,doi,title,abstract_inverted_index,publication_year,open_access,primary_location,authorships",
            }
            r = requests.get(base, params=params, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(5); continue
            r.raise_for_status()
            data = r.json()
            results = data.get("results", [])
            for w in results:
                pl = w.get("primary_location") or {}
                src = pl.get("source") or {}
                oa = w.get("open_access") or {}
                rows.append({
                    "arxiv_id": None,
                    "doi": _normalize_doi(w.get("doi")),
                    "openalex_id": w.get("id"),
                    "title": (w.get("title") or "").strip(),
                    "abstract": _reconstruct_abstract_oa(w.get("abstract_inverted_index"))[:5000],
                    "authors": [a.get("author", {}).get("display_name") for a in (w.get("authorships") or [])],
                    "year": w.get("publication_year"),
                    "venue": src.get("display_name"),
                    "oa_pdf_url": oa.get("oa_url"),
                    "is_oa": oa.get("is_oa", False),
                    "discovered_via": "openalex",
                    "discovered_query": q,
                })
            cursor = data.get("meta", {}).get("next_cursor")
            collected += len(results)
            if not cursor or not results:
                break
            time.sleep(0.1)
        print(f"  '{q}': {collected} resultados")
    return pd.DataFrame(rows)

In [9]:
openalex_path = RAW_DIR / "openalex_papers.parquet"

if openalex_path.exists():
    print(f"Cache: {openalex_path}")
    df_openalex = pd.read_parquet(openalex_path)
else:
    print("Buscando no OpenAlex...")
    df_openalex = search_openalex(KEYWORD_QUERIES, max_per_query=1500, email=USER_EMAIL)
    df_openalex.to_parquet(openalex_path, index=False)
    print(f"Salvo em {openalex_path}")

print(f"\nTotal de linhas (com duplicatas entre queries): {len(df_openalex)}")
print(f"Únicos por openalex_id: {df_openalex['openalex_id'].nunique()}")
print(f"Com PDF open access: {int(df_openalex['is_oa'].sum())}")
df_openalex.head(3)

Buscando no OpenAlex...
  'class imbalance': 1600 resultados
  'imbalanced classification': 1600 resultados
  'imbalanced learning': 1600 resultados
  'imbalanced data': 1600 resultados
  'minority class oversampling': 1600 resultados
  'undersampling minority class': 1600 resultados
  'cost-sensitive learning': 1600 resultados
  'class reweighting': 1600 resultados
  'class-balanced loss': 1600 resultados
  'SMOTE': 1600 resultados
  'ADASYN': 1600 resultados
  'focal loss': 1600 resultados
  'long-tail recognition': 1600 resultados
  'long-tailed classification': 1600 resultados
Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\raw\openalex_papers.parquet

Total de linhas (com duplicatas entre queries): 22400
Únicos por openalex_id: 16869
Com PDF open access: 14551


,arxiv_id,doi,openalex_id,title,abstract,authors,year,venue,oa_pdf_url,is_oa,discovered_via,discovered_query
0,None,10.3233/ida-2002-6504,https://openalex.org/W1941659294,The class imbalance problem: A systematic study1,"In machine learning problems, differences in p...","[Nathalie Japkowicz, Shaju Stephen]",2002.0,Intelligent Data Analysis,None,False,openalex,class imbalance
1,None,10.1186/s40537-019-0192-5,https://openalex.org/W2936503027,Survey on deep learning with class imbalance,The purpose of this study is to examine existi...,"[Justin Johnson, Taghi M. Khoshgoftaar]",2019.0,Journal Of Big Data,https://journalofbigdata.springeropen.com/trac...,True,openalex,class imbalance
2,None,10.1109/tsmcb.2008.2007853,https://openalex.org/W2104167780,Exploratory Undersampling for Class-Imbalance ...,Undersampling is a popular method in dealing w...,"[Xuying Liu, Jianxin Wu, Zhihua Zhou]",2008.0,IEEE Transactions on Systems Man and Cyberneti...,None,False,openalex,class imbalance


## Fonte 3: Semantic Scholar

O Semantic Scholar retorna o campo `openAccessPdf.url` diretamente, indicando se o PDF é acessível.

In [10]:
def search_semantic_scholar(queries, max_per_query=1500, api_key=None):
    base = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
    headers = {"x-api-key": api_key} if api_key else {}
    fields = "paperId,corpusId,externalIds,title,abstract,year,authors,openAccessPdf,venue,fieldsOfStudy"
    rows = []
    for q in queries:
        token = None
        collected = 0
        while collected < max_per_query:
            params = {"query": q, "fields": fields}
            if token:
                params["token"] = token
            try:
                r = requests.get(base, params=params, headers=headers, timeout=60)
            except requests.RequestException as e:
                print(f"  '{q}': erro de rede — {e}; pausando 10s")
                time.sleep(10); continue
            if r.status_code == 429:
                time.sleep(10); continue
            if r.status_code >= 400:
                print(f"  '{q}': HTTP {r.status_code} — {r.text[:200]}")
                break
            data = r.json()
            results = data.get("data") or []
            for p in results:
                ext = p.get("externalIds") or {}
                oa_pdf = p.get("openAccessPdf") or {}
                rows.append({
                    "arxiv_id": ext.get("ArXiv"),
                    "doi": _normalize_doi(ext.get("DOI")),
                    "s2_paper_id": p.get("paperId"),
                    "title": (p.get("title") or "").strip(),
                    "abstract": (p.get("abstract") or "").strip(),
                    "authors": [a.get("name") for a in (p.get("authors") or [])],
                    "year": p.get("year"),
                    "venue": p.get("venue"),
                    "fields_of_study": p.get("fieldsOfStudy"),
                    "oa_pdf_url": oa_pdf.get("url"),
                    "is_oa": bool(oa_pdf.get("url")),
                    "discovered_via": "s2",
                    "discovered_query": q,
                })
            collected += len(results)
            token = data.get("token")
            if not token or not results:
                break
            time.sleep(1.0 if not api_key else 0.1)
        print(f"  '{q}': {collected} resultados")
    return pd.DataFrame(rows)

In [12]:
s2_path = RAW_DIR / "s2_papers.parquet"

if s2_path.exists():
    print(f"Cache: {s2_path}")
    df_s2 = pd.read_parquet(s2_path)
else:
    print("Buscando no Semantic Scholar...")
    df_s2 = search_semantic_scholar(KEYWORD_QUERIES, max_per_query=1500)
    df_s2.to_parquet(s2_path, index=False)
    print(f"Salvo em {s2_path}")

print(f"\nTotal de linhas (com duplicatas entre queries): {len(df_s2)}")
print(f"Únicos por s2_paper_id: {df_s2['s2_paper_id'].nunique()}")
print(f"Com PDF open access: {int(df_s2['is_oa'].sum())}")
df_s2.head(3)

Buscando no Semantic Scholar...
  'class imbalance': 2000 resultados
  'imbalanced classification': 2000 resultados
  'imbalanced learning': 2000 resultados
  'imbalanced data': 2000 resultados
  'minority class oversampling': 2000 resultados
  'undersampling minority class': 741 resultados
  'cost-sensitive learning': 2000 resultados
  'class reweighting': 762 resultados
  'class-balanced loss': 2000 resultados
  'SMOTE': 2000 resultados
  'ADASYN': 1355 resultados
  'focal loss': 2000 resultados
  'long-tail recognition': 1883 resultados
  'long-tailed classification': 2000 resultados
Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\raw\s2_papers.parquet

Total de linhas (com duplicatas entre queries): 24741
Únicos por s2_paper_id: 19406
Com PDF open access: 6814


,arxiv_id,doi,s2_paper_id,title,abstract,authors,year,venue,fields_of_study,oa_pdf_url,is_oa,discovered_via,discovered_query
0,None,10.1109/iisec69317.2026.11418517,0000f7b97914da30c259a1d2acc1903371d7cfde,Deep Learning Approaches for Pneumonia Detecti...,Artificial intelligence (AI) has become increa...,"[Zeynep Bolukbasi, Hilal Yurtoglu, Emin Kugu, ...",2026.0,2026 5th International Informatics and Softwar...,None,,False,s2,class imbalance
1,None,10.1145/1871871.1871886,00020a555b2fc0a7dadc0aa0111bca734e383a4d,Effect of classifiers in consensus feature ran...,,"[Shobeir Fakhraei, H. Soltanian-Zadeh, F. Foto...",2010.0,Data and Text Mining in Bioinformatics,[Computer Science],http://www.cs.wayne.edu/shobeir/papers/2010_DT...,True,s2,class imbalance
2,None,10.33920/med-15-2404-02,00059c2c190743e3d928509966d4fcce63162924,Some etiological features of spontaneous abort...,The review article examines some etiological f...,"[M. S. Danilova, R. Bontsevich, M. L. Maksimov]",2024.0,Hirurg (Surgeon),None,,False,s2,class imbalance


## Unificação e deduplicação

O mesmo paper costuma aparecer em mais de uma fonte sob IDs diferentes. Unificamos pela ordem de prioridade:

1. **arXiv ID** (mais confiável quando existe)
2. **DOI** normalizado (lowercase, sem prefixo URL)
3. **Título** normalizado (lowercase, alfanumérico apenas, mínimo 10 caracteres úteis)

Para cada paper único, agregamos os IDs externos, lista de fontes e queries que o retornaram. Para os demais campos (title, abstract, year, ...) mantemos o **primeiro valor não-nulo** encontrado entre as fontes.

In [5]:
def _normalize_arxiv_id(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    x = str(x).strip()
    x = re.sub(r"v\d+$", "", x)  # remove sufixo de versão
    return x or None

def _normalize_title_key(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return None
    k = re.sub(r"\W+", "", str(t).lower())
    return k if len(k) >= 10 else None

EXPECTED_COLS = [
    "arxiv_id", "doi", "s2_paper_id", "openalex_id",
    "title", "abstract", "authors", "year", "venue",
    "oa_pdf_url", "is_oa",
    "categories", "fields_of_study",
    "discovered_via", "discovered_query",
]

def unify_and_dedupe(dfs):
    parts = []
    for df in dfs:
        if df is None or len(df) == 0:
            continue
        d = df.copy()
        for c in EXPECTED_COLS:
            if c not in d.columns:
                d[c] = None
        parts.append(d[EXPECTED_COLS])
    if not parts:
        return pd.DataFrame(columns=EXPECTED_COLS)
    unified = pd.concat(parts, ignore_index=True)

    # Normaliza identificadores
    unified["arxiv_id"] = unified["arxiv_id"].apply(_normalize_arxiv_id)
    unified["doi"] = unified["doi"].apply(_normalize_doi)
    unified["title_key"] = unified["title"].apply(_normalize_title_key)

    # merge_key = arxiv_id > doi > title_key
    unified["merge_key"] = unified["arxiv_id"]
    mask = unified["merge_key"].isna()
    unified.loc[mask, "merge_key"] = unified.loc[mask, "doi"]
    mask = unified["merge_key"].isna()
    unified.loc[mask, "merge_key"] = unified.loc[mask, "title_key"]

    before = len(unified)
    unified = unified[unified["merge_key"].notna()].copy()
    print(f"Linhas sem chave de merge descartadas: {before - len(unified)}")

    def agg_group(g):
        out = {}
        for col in EXPECTED_COLS:
            if col in ("discovered_via", "discovered_query"):
                continue
            nn = [v for v in g[col].tolist()
                  if v is not None and not (isinstance(v, float) and pd.isna(v))]
            out[col] = nn[0] if nn else None
        out["discovered_via"] = sorted(set(v for v in g["discovered_via"].tolist() if v))
        out["discovered_queries"] = sorted(set(v for v in g["discovered_query"].tolist() if v))
        out["n_sources"] = len(out["discovered_via"])
        return pd.Series(out)

    deduped = unified.groupby("merge_key", sort=False).apply(agg_group).reset_index()
    return deduped

def generate_paper_id(row):
    if row.get("arxiv_id"):
        return f"arxiv:{row['arxiv_id']}"
    if row.get("doi"):
        return f"doi:{row['doi']}"
    if row.get("s2_paper_id"):
        return f"s2:{row['s2_paper_id']}"
    oa = row.get("openalex_id")
    if oa:
        return f"oa:{str(oa).split('/')[-1]}"
    h = hashlib.md5(f"{row.get('title', '')}_{row.get('year', '')}".encode()).hexdigest()[:12]
    return f"hash:{h}"

In [6]:
unified_path = PROCESSED_DIR / "papers_unified.parquet"

if unified_path.exists():
    print(f"Cache: {unified_path}")
    df_unified = pd.read_parquet(unified_path)
else:
    print("Unificando e deduplicando...")
    df_unified = unify_and_dedupe([df_arxiv, df_openalex, df_s2])
    df_unified["paper_id"] = df_unified.apply(generate_paper_id, axis=1)
    df_unified.to_parquet(unified_path, index=False)
    print(f"Salvo em {unified_path}")

print(f"\nPapers únicos: {len(df_unified)}")
print("\nDistribuição por número de fontes (overlap):")
print(df_unified["n_sources"].value_counts().sort_index())
print(f"\nCom PDF open access: {int(df_unified['is_oa'].fillna(False).sum())}")
print(f"\nPor ano (últimos 10):")
print(df_unified["year"].value_counts().sort_index().tail(10))

Cache: c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_unified.parquet

Papers únicos: 36859

Distribuição por número de fontes (overlap):
n_sources
1    32092
2     4767
Name: count, dtype: int64

Com PDF open access: 19756

Por ano (últimos 10):
year
2017.0    1086
2018.0    1502
2019.0    2033
2020.0    2575
2021.0    2926
2022.0    3108
2023.0    3575
2024.0    3904
2025.0    4939
2026.0    1503
Name: count, dtype: int64


## Filtro 2: ranking por embedding semântico

Juntamos title + abstract de cada paper e os `EMBEDDING_QUERIES` com `sentence-transformers/all-MiniLM-L6-v2` (modelo viável de rodar em cpu). Calculamos similaridade cosseno entre cada paper e cada query.

Mantemos top_k papers. K=3000 é provavelmente bem generoso: dá folga para o classificador LLM no próximo notebook descartar falsos positivos sem perder relevantes na borda.

> **Filtro mínimo:** papers com texto utilizável muito curto (`< 50 caracteres` em title+abstract) são descartados aqui. Sem texto suficiente, o embedding é pouco informativo.

In [7]:
def rank_by_embedding(df, queries=EMBEDDING_QUERIES, model_name="sentence-transformers/all-MiniLM-L6-v2", top_k=3000, batch_size=64):
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    df = df.copy()
    df["text_for_embedding"] = (
        df["title"].fillna("") + ". " + df["abstract"].fillna("")
    ).str.slice(0, 2000)
    mask = df["text_for_embedding"].str.len() >= 50
    print(f"Papers com texto utilizável: {int(mask.sum())} / {len(df)}")
    df_valid = df[mask].copy().reset_index(drop=True)

    print(f"Carregando modelo {model_name}...")
    model = SentenceTransformer(model_name)

    print(f"Codificando {len(df_valid)} papers...")
    paper_embs = model.encode(
        df_valid["text_for_embedding"].tolist(),
        show_progress_bar=True, batch_size=batch_size,
        convert_to_numpy=True,
    )
    query_embs = model.encode(queries, convert_to_numpy=True)
    sims = cosine_similarity(paper_embs, query_embs)
    df_valid["emb_score_max"] = sims.max(axis=1)
    df_valid["emb_score_mean"] = sims.mean(axis=1)

    df_valid = df_valid.sort_values("emb_score_max", ascending=False).reset_index(drop=True)
    df_valid["rank"] = df_valid.index + 1
    return df_valid.head(top_k).drop(columns=["text_for_embedding"])

In [8]:
ranked_path = PROCESSED_DIR / "papers_ranked.parquet"

if ranked_path.exists():
    print(f"Cache: {ranked_path}")
    df_ranked = pd.read_parquet(ranked_path)
else:
    df_ranked = rank_by_embedding(df_unified, top_k=TOP_K)
    df_ranked.to_parquet(ranked_path, index=False)
    print(f"Salvo em {ranked_path}")

print(f"\nTop-{len(df_ranked)} papers selecionados")
print("\nDistribuição do score (max):")
print(df_ranked["emb_score_max"].describe())
print(f"\nCom PDF open access no top-K: {int(df_ranked['is_oa'].fillna(False).sum())}")
print("\nTop-10 por score (sanity check manual):")
df_ranked.head(10)[["rank", "emb_score_max", "year", "title"]]

Papers com texto utilizável: 36242 / 36859
Carregando modelo sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5460.02it/s]


Codificando 36242 papers...


Batches: 100%|██████████| 567/567 [25:22<00:00,  2.69s/it]


Salvo em c:\Users\fredb\Desktop\Faculdade\CC\2026.1\causal\project\data\processed\papers_ranked.parquet

Top-3000 papers selecionados

Distribuição do score (max):
count    3000.000000
mean        0.625158
std         0.036538
min         0.575775
25%         0.594689
50%         0.618255
75%         0.649332
max         0.776774
Name: emb_score_max, dtype: float64

Com PDF open access no top-K: 1405

Top-10 por score (sanity check manual):


,rank,emb_score_max,year,title
0,1,0.776774,2021.0,Experimental Comparison of Classification Meth...
1,2,0.753539,2022.0,An empirical evaluation of sampling methods fo...
2,3,0.752201,2020.0,Classification Accuracy Comparison for Imbalan...
3,4,0.752127,2024.0,"Performance of Random Oversampling, Random Und..."
4,5,0.751496,2025.0,Advanced Re-Sampling Techniques for Multi-Clas...
5,6,0.749815,2024.0,Enhanced Support Vectors based Oversampling Me...
6,7,0.748733,2016.0,Beyond the Boundaries of SMOTE: A Framework fo...
7,8,0.747164,2013.0,Adaptive Oversampling for Imbalanced Data Clas...
8,9,0.746176,2025.0,Tackling Minority Class Detection: A Study on ...
9,10,0.745438,2022.0,Fraud Detection Using Large-scale Imbalance Da...
